In [9]:
import json
import os

import requests
from dotenv import load_dotenv

In [10]:
load_dotenv()  # Load environment variables from .env file
max_batch_size = int(os.getenv("MAX_BATCH_INGEST_SIZE", 10))

In [11]:
wiki_link_array = [
    "https://en.wikipedia.org/wiki/Mathematics",
    "https://en.wikipedia.org/wiki/Number_theory",
    "https://en.wikipedia.org/wiki/Calculus",
    "https://en.wikipedia.org/wiki/Linear_algebra",
    "https://en.wikipedia.org/wiki/Euclidean_geometry",
    "https://en.wikipedia.org/wiki/Topology",
    "https://en.wikipedia.org/wiki/Probability_theory",
    "https://en.wikipedia.org/wiki/Statistics",
    "https://en.wikipedia.org/wiki/Combinatorics",
    "https://en.wikipedia.org/wiki/Graph_theory",
    "https://en.wikipedia.org/wiki/Set_theory",
    "https://en.wikipedia.org/wiki/Logic",
    "https://en.wikipedia.org/wiki/Number_system",
    "https://en.wikipedia.org/wiki/Algebraic_geometry",
    "https://en.wikipedia.org/wiki/Differential_equations",
    "https://en.wikipedia.org/wiki/Mathematical_analysis",
    "https://en.wikipedia.org/wiki/Functional_analysis",
    "https://en.wikipedia.org/wiki/Complex_analysis",
    "https://en.wikipedia.org/wiki/Real_analysis",
    "https://en.wikipedia.org/wiki/Numerical_analysis",
]

base_url = "http://localhost:8000"

In [4]:
print("Number of links to process: ", len(wiki_link_array))

Number of links to process:  20


In [5]:
# Split the list into sublists of at most 10 strings
batches = [wiki_link_array[i : i + max_batch_size] for i in range(0, len(wiki_link_array), max_batch_size)]

# Print the result
for idx, batch in enumerate(batches):
    print(f"Batch {idx + 1} (Size {len(batch)}): {batch}")

Batch 1 (Size 10): ['https://en.wikipedia.org/wiki/Mathematics', 'https://en.wikipedia.org/wiki/Number_theory', 'https://en.wikipedia.org/wiki/Calculus', 'https://en.wikipedia.org/wiki/Linear_algebra', 'https://en.wikipedia.org/wiki/Euclidean_geometry', 'https://en.wikipedia.org/wiki/Topology', 'https://en.wikipedia.org/wiki/Probability_theory', 'https://en.wikipedia.org/wiki/Statistics', 'https://en.wikipedia.org/wiki/Combinatorics', 'https://en.wikipedia.org/wiki/Graph_theory']
Batch 2 (Size 10): ['https://en.wikipedia.org/wiki/Set_theory', 'https://en.wikipedia.org/wiki/Logic', 'https://en.wikipedia.org/wiki/Number_system', 'https://en.wikipedia.org/wiki/Algebraic_geometry', 'https://en.wikipedia.org/wiki/Differential_equations', 'https://en.wikipedia.org/wiki/Mathematical_analysis', 'https://en.wikipedia.org/wiki/Functional_analysis', 'https://en.wikipedia.org/wiki/Complex_analysis', 'https://en.wikipedia.org/wiki/Real_analysis', 'https://en.wikipedia.org/wiki/Numerical_analysis']


In [ ]:
job_ids = []
for idx, batch in enumerate(batches):
    url_array = []
    for url in batch:
        elements = url.split("/")
        title = elements[-1]
        url_array.append({"url": url, "title": title})
    payload = {"documents": url_array}
    response = requests.post(f"{base_url}/api/v1/docsets/mathematics/documents", json=payload)
    if response.status_code == 202:
        job_id = response.json().get("main_job_id")
        job_ids.append(job_id)
        print(f"Batch {idx + 1} submitted successfully. Job ID: {job_id}")
    else:
        print(f"Failed to submit batch {idx + 1}. Status code: {response.status_code}, Response: {response.text}")

print("All batches submitted. Job IDs:", job_ids)

Batch 1 submitted successfully. Job ID: 3f51395e-a9f9-49b6-938e-164052d340d5
Batch 2 submitted successfully. Job ID: 11657a70-272e-4a48-af48-a08bef123b83
All batches submitted. Job IDs: ['3f51395e-a9f9-49b6-938e-164052d340d5', '11657a70-272e-4a48-af48-a08bef123b83']


In [12]:
for job_id in job_ids:
    response = requests.get(f"{base_url}/api/v1/documents/status/{job_id}")
    if response.status_code == 200:
        response_json = response.json()
        status = response_json["status"]
        print(f"Job ID: {job_id}, Status: {status}")
        print(f"Response received: {json.dumps(response_json, indent=4)}")
    else:
        print(f"Failed to get status for Job ID: {job_id}.")
        print(f"Status code: {response.status_code}, Response: {response.text}")

Job ID: 3f51395e-a9f9-49b6-938e-164052d340d5, Status: COMPLETED
Response received: {
    "main_job_id": "3f51395e-a9f9-49b6-938e-164052d340d5",
    "status": "COMPLETED",
    "overall_progress_percentage": 100,
    "total_jobs": 10,
    "completed_jobs": 10,
    "failed_jobs": 0,
    "jobs": [
        {
            "url": "https://en.wikipedia.org/wiki/Topology",
            "doc_id": "5695515c-f57b-4abd-8a7c-f3ed1dbe1b84",
            "job_id": "e0b62dec-b06a-4779-9e6f-03b4ba3e10c3",
            "status": "INDEXED",
            "progress_percentage": 100,
            "error_message": null
        },
        {
            "url": "https://en.wikipedia.org/wiki/Probability_theory",
            "doc_id": "11bf4511-6d85-442e-80ee-e3b890ed849d",
            "job_id": "73bb4e64-4228-4b11-930b-910fdcfc713e",
            "status": "INDEXED",
            "progress_percentage": 100,
            "error_message": null
        },
        {
            "url": "https://en.wikipedia.org/wiki/Combinato

In [13]:
response = requests.get(f"{base_url}/api/v1/docsets?limit=20&offset=0")

response_json = response.json()
print(f"Response received: {json.dumps(response_json, indent=4)}")

Response received: {
    "total": 1,
    "limit": 20,
    "offset": 0,
    "items": [
        {
            "name": "mathematics",
            "document_count": 20,
            "created_at": "2026-09-16T11:38:25.769621Z",
            "updated_at": "2026-09-16T11:38:28.045433Z"
        }
    ]
}


In [ ]:
response = requests.get(f"{base_url}/api/v1/docsets/mathematics/documents")

response_json = response.json()
print(f"Response received: {json.dumps(response_json, indent=4)}")